# RoBERTa Emotion Classification - Training Notebook

This notebook trains a RoBERTa model with LoRA for 6-way emotion classification.

**Features:**
- Weighted loss to handle class imbalance
- LoRA fine-tuning for efficiency
- GPU/TPU acceleration
- Model checkpointing and saving


## 1. Setup and Installation


In [ ]:
# Install dependencies
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install -q transformers datasets peft accelerate scikit-learn tqdm pandas numpy matplotlib seaborn


In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# Configuration
CANONICAL_EMOTIONS = ["anger", "disgust", "fear", "happy", "neutral", "sad"]

# Emotion mapping from GoEmotions to canonical 6-way
GOEMOTIONS_TO_CANONICAL = {
    "anger": "anger",
    "annoyance": "anger",
    "disapproval": "anger",
    "disgust": "disgust",
    "fear": "fear",
    "nervousness": "fear",
    "joy": "happy",
    "amusement": "happy",
    "excitement": "happy",
    "gratitude": "happy",
    "love": "happy",
    "optimism": "happy",
    "relief": "happy",
    "pride": "happy",
    "admiration": "happy",
    "sadness": "sad",
    "disappointment": "sad",
    "embarrassment": "sad",
    "grief": "sad",
    "remorse": "sad",
    "neutral": "neutral",
}

CANONICAL_SET = set(CANONICAL_EMOTIONS)

def map_goemotions_labels(labels):
    """Map fine-grained GoEmotions labels to canonical 6-way emotions."""
    if not labels:
        return None
    
    mapped = []
    for label in labels:
        if label in GOEMOTIONS_TO_CANONICAL:
            canonical = GOEMOTIONS_TO_CANONICAL[label]
            if canonical in CANONICAL_SET:
                mapped.append(canonical)
    
    if not mapped:
        return None
    
    # If multiple emotions, prefer: anger > disgust > fear > sad > happy > neutral
    priority = ["anger", "disgust", "fear", "sad", "happy", "neutral"]
    for p in priority:
        if p in mapped:
            return p
    
    return mapped[0]


## 3. Load and Prepare GoEmotions Dataset


In [ ]:
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm

print("Loading GoEmotions dataset...")
ds = load_dataset("go_emotions", "raw", split="train")
df = ds.to_pandas()
print(f"Loaded {len(df)} examples")
print(f"Columns: {df.columns.tolist()[:10]}...")


In [ ]:
# Extract labels from emotion columns
emotion_cols = [col for col in df.columns if col in GOEMOTIONS_TO_CANONICAL]
print(f"Found {len(emotion_cols)} emotion columns")

def get_active_labels(row):
    return [col for col in emotion_cols if row[col] == 1]

df["raw_labels"] = df.apply(get_active_labels, axis=1)
df["canonical_label"] = df["raw_labels"].apply(map_goemotions_labels)

# Filter out None labels
df_filtered = df[df["canonical_label"].notna()].copy()
print(f"After filtering: {len(df_filtered)} examples")

# Create final dataset
df_final = pd.DataFrame({
    "text": df_filtered["text"].values,
    "label": df_filtered["canonical_label"].values
})

print(f"\nFinal dataset size: {len(df_final)}")
print(f"\nClass distribution:")
print(df_final["label"].value_counts())


In [ ]:
# Visualize class distribution
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.countplot(data=df_final, x="label", order=df_final["label"].value_counts().index)
plt.title("GoEmotions 6-class Label Distribution")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Calculate class weights for weighted loss
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = df_final["label"].values
unique_labels = np.unique(labels)
class_weights = compute_class_weight('balanced', classes=unique_labels, y=labels)
class_weight_dict = dict(zip(unique_labels, class_weights))

print("\nClass weights for weighted loss:")
for label, weight in sorted(class_weight_dict.items()):
    print(f"  {label:12s} {weight:.4f}")


## 4. Model Setup


In [ ]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from typing import List

class TextEmotionDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# Create label encoding
label2id = {label: i for i, label in enumerate(CANONICAL_EMOTIONS)}
id2label = {i: label for label, i in label2id.items()}

# Stratified train/val split
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, val_idx = next(splitter.split(df_final["text"], df_final["label"]))

train_df = df_final.iloc[train_idx].reset_index(drop=True)
val_df = df_final.iloc[val_idx].reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")

# Encode labels
train_labels = [label2id[str(label)] for label in train_df["label"]]
val_labels = [label2id[str(label)] for label in val_df["label"]]

# Load tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

# Create datasets
train_ds = TextEmotionDataset(train_df["text"].tolist(), train_labels, tokenizer, max_length=128)
val_ds = TextEmotionDataset(val_df["text"].tolist(), val_labels, tokenizer, max_length=128)

# Create data loaders
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")


In [ ]:
from peft import LoraConfig, get_peft_model
import warnings
warnings.filterwarnings("ignore")

# Load base model
print("Loading RoBERTa base model...")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(CANONICAL_EMOTIONS),
    id2label=id2label,
    label2id=label2id
)

# Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
)
model = get_peft_model(model, lora_config)

# Freeze base model, keep LoRA and classifier trainable
for name, param in model.named_parameters():
    if "classifier" in name or "lora_" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

model = model.to(device)
print(f"Model moved to {device}")


In [ ]:
from torch import nn
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Create class weights tensor
weights_list = [class_weight_dict[label] for label in CANONICAL_EMOTIONS]
class_weights_tensor = torch.tensor(weights_list, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer (only trainable parameters)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=2e-5, weight_decay=0.01)

# Learning rate scheduler
num_epochs = 5
total_steps = len(train_loader) * num_epochs
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: 2e-5")
print(f"  Batch size: 32")
print(f"  Total steps: {total_steps}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Using weighted loss: Yes")


In [ ]:
def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0
    
    from tqdm import tqdm
    pbar = tqdm(dataloader, desc="Training")
    
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        
        outputs = model(**batch)
        logits = outputs.logits
        
        # Use weighted loss
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_logits = []
    all_labels = []
    
    from tqdm import tqdm
    pbar = tqdm(dataloader, desc="Validating")
    
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        
        outputs = model(**batch)
        logits = outputs.logits
        loss = criterion(logits, labels)
        
        total_loss += loss.item()
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(dataloader)
    logits_arr = np.concatenate(all_logits, axis=0)
    labels_arr = np.concatenate(all_labels, axis=0)
    
    preds = logits_arr.argmax(axis=-1)
    acc = accuracy_score(labels_arr, preds)
    macro_f1 = f1_score(labels_arr, preds, average="macro")
    
    # Per-class F1
    per_class_f1 = f1_score(labels_arr, preds, average=None, labels=range(len(CANONICAL_EMOTIONS)))
    
    return avg_loss, acc, macro_f1, per_class_f1


In [ ]:
import os
from datetime import datetime

# Training history
history = {
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": []
}

best_macro_f1 = -1.0
patience = 3
epochs_without_improvement = 0

print("=" * 60)
print("Starting Training")
print("=" * 60)

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs}")
    
    # Train
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    
    # Evaluate
    val_loss, val_acc, val_macro_f1, per_class_f1 = evaluate(model, val_loader, criterion, device)
    
    # Update history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)
    
    # Print metrics
    print(f"\n  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Val Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")
    print(f"  Val Macro-F1: {val_macro_f1:.4f}")
    print(f"\n  Per-class F1:")
    for i, emotion in enumerate(CANONICAL_EMOTIONS):
        print(f"    {emotion:12s} {per_class_f1[i]:.4f}")
    
    # Check for improvement
    is_best = val_macro_f1 > best_macro_f1
    if is_best:
        best_macro_f1 = val_macro_f1
        epochs_without_improvement = 0
        print(f"\n  ✓ New best macro-F1: {best_macro_f1:.4f}")
        
        # Save best model
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_macro_f1": best_macro_f1,
            "val_acc": val_acc,
            "per_class_f1": per_class_f1.tolist(),
            "label2id": label2id,
            "id2label": id2label,
        }
        torch.save(checkpoint, "best_model.pt")
        print(f"  Saved best model checkpoint")
    else:
        epochs_without_improvement += 1
        print(f"\n  No improvement ({epochs_without_improvement}/{patience})")
    
    # Early stopping
    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping triggered after {epoch} epochs")
        break

print("\n" + "=" * 60)
print("Training Complete!")
print("=" * 60)
print(f"Best Macro-F1: {best_macro_f1:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history["val_acc"], label="Val Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].legend()
axes[1].grid(True)

# Macro-F1
axes[2].plot(history["val_macro_f1"], label="Val Macro-F1")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Macro-F1")
axes[2].set_title("Validation Macro-F1")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()


## 8. Test Model on Sample Sentences


In [ ]:
# Load best model
checkpoint = torch.load("best_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Test sentences
test_sentences = [
    "I'm so happy and excited!",
    "This is absolutely ridiculous and unacceptable!",
    "That's disgusting and repulsive!",
    "I'm terrified and scared!",
    "I feel sad and disappointed.",
    "The weather is nice today.",
]

print("Testing model predictions:\n")
print("=" * 60)

for text in test_sentences:
    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).squeeze()
        pred_id = probs.argmax().item()
        pred_label = id2label[pred_id]
        pred_prob = probs[pred_id].item()
    
    print(f"\nText: '{text}'")
    print(f"Predicted: {pred_label.upper()} (prob: {pred_prob:.3f})")
    print(f"Top 3:")
    top3 = torch.topk(probs, 3)
    for i, (prob, idx) in enumerate(zip(top3.values, top3.indices)):
        print(f"  {id2label[idx.item()]:12s} {prob.item():.3f}")

print("\n" + "=" * 60)


## 9. Download Model


In [ ]:
from google.colab import files

# Download the best model checkpoint
files.download("best_model.pt")
print("\n✓ Model downloaded!")
print("\nTo use this model in your local project:")
print("1. Place 'best_model.pt' in: data/processed/models/checkpoints/")
print("2. Run your test script: python scripts/test_text_model_live.py")
